In [ ]:
!pip install gradio

In [ ]:
# --- IMPORTS ---
from google.colab import files
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision import models
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import time
import random

import torch
import torchvision.models as models
from torchvision import transforms
from PIL import Image

# --- 1. UPLOAD IMAGE + METADATA ---
# Upload image
uploaded = files.upload()
for filename in uploaded.keys():
    print(f"✅ Uploaded file: {filename}")
    image_path = filename

# Dropdowns for metadata
body_part_options = ['Face', 'Arm', 'Back', 'Chest', 'Leg', 'Scalp', 'Abdomen', 'Other']
body_part_dropdown = widgets.Dropdown(options=body_part_options, description='Body Part:')
timeline_options = ['Appeared recently', 'Present for months', 'Present for years', 'Changing recently', 'No change']
timeline_dropdown = widgets.Dropdown(options=timeline_options, description='Timeline:')
display(body_part_dropdown, timeline_dropdown)

def get_metadata():
    selected_body_part = body_part_dropdown.value
    selected_timeline = timeline_dropdown.value
    return selected_body_part, selected_timeline

# --- 2. LOAD MODEL ---
# Upload model file
uploaded = files.upload()
for filename in uploaded.keys():
    model_path = filename

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Saving enhanced_skin_lesion_classifier2 (1).pth to enhanced_skin_lesion_classifier2 (1) (5).pth
✅ Uploaded file: enhanced_skin_lesion_classifier2 (1) (5).pth


Dropdown(description='Body Part:', options=('Face', 'Arm', 'Back', 'Chest', 'Leg', 'Scalp', 'Abdomen', 'Other'…

Dropdown(description='Timeline:', options=('Appeared recently', 'Present for months', 'Present for years', 'Ch…

In [ ]:
# Define label mapping
class_labels = {
    0: "Superficial spreading melanoma",
    1: "Nodular melanoma",
    2: "Lentigo maligna melanoma",
    3: "Basal cell carcinoma",
    4: "Squamous cell carcinoma",
    5: "Actinic keratosis",
    6: "Melanocytic nevus (mole)",
    7: "Dermatofibroma",
    8: "Seborrheic keratosis",
    9: "Vascular lesions (hemangiomas)",
    10: "Fungal infections (tinea)",
    11: "Viral warts",
    12: "Bacterial infections",
    13: "Eczema",
    14: "Psoriasis",
    15: "Acne",
    16: "Rosacea"
}

# Setup model
# In your Gradio app
def load_skin_model():
    try:
        # Try loading as state_dict
        model = models.resnet50(weights=None)
        num_ftrs = model.fc.in_features
        model.fc = torch.nn.Linear(num_ftrs, 10)
        model.load_state_dict(torch.load('enhanced_skin_lesion_classifier2.pth',
                                        map_location='cpu'))
    except:
        try:
            # Try loading with pickle_module=torch.serialization.pickle
            import pickle
            model = torch.load('enhanced_skin_lesion_classifier2.pth',
                              map_location='cpu',
                              pickle_module=pickle)
        except:
            # Fallback to a simple model for demo purposes
            model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            model.eval()
    model.eval()
    return model

# --- 3. IMAGE TRANSFORM + PREDICTION ---
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def predict_skin_condition(image_path):
    try:
        model = load_skin_model()

        # Image transformation - same as in your testing code
        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ])

        # Process the image
        image = Image.open(image_path).convert('RGB')
        image_tensor = transform(image).unsqueeze(0)

        # Make prediction
        with torch.no_grad():
            output = model(image_tensor)
            probs = torch.nn.functional.softmax(output, dim=1)[0]

        # Convert to dictionary of results
        results = {}
        for i, prob in enumerate(probs):
            results[class_labels[i]] = float(prob) * 100

        # Sort by confidence
        results = dict(sorted(results.items(), key=lambda x: x[1], reverse=True))
        return results

    except Exception as e:
        print(f"Error in prediction: {str(e)}")
        # Return fallback values in case of errors
        return {
            "Error": 100,
            "Model prediction failed": 0
        }

# --- 4. ANALYZE + OUTPUT RESULTS ---
# Your taxonomy is already defined in the model
def analyze_image(image, body_part, timeline):
    if image is None:
        return ("Please upload an image.", "", "", "")

    try:
        # Load the model
        model = load_skin_model()

        # Process the image
        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ])

        img = Image.open(image).convert('RGB')
        img_tensor = transform(img).unsqueeze(0)

        # Get prediction
        with torch.no_grad():
            output = model(img_tensor)
            probs = torch.nn.functional.softmax(output, dim=1)[0]

        # Create a flattened list of all conditions from the taxonomy
        all_conditions = []
        for category in taxonomy:
            for subcategory in taxonomy[category]:
                for condition in taxonomy[category][subcategory]:
                    all_conditions.append(condition)

        # Match probabilities with conditions
        results = {}
        for i, prob in enumerate(probs):
            if i < len(all_conditions):
                results[all_conditions[i]] = float(prob) * 100

        # Sort by confidence
        results = dict(sorted(results.items(), key=lambda x: x[1], reverse=True))

        # Format results with category information
        results_text = "AI Analysis Results:\n"

        # Get the top prediction and its category
        top_condition = list(results.keys())[0]
        top_category = None
        top_subcategory = None

        for category in taxonomy:
            for subcategory in taxonomy[category]:
                if top_condition in taxonomy[category][subcategory]:
                    top_category = category
                    top_subcategory = subcategory
                    break

        # Add category information
        if top_category and top_subcategory:
            results_text += f"Category: {top_category} / {top_subcategory}\n\n"

        # Add top 3 predictions
        results_text += "Top Predictions:\n"
        for condition, confidence in list(results.items())[:3]:
            results_text += f"• {condition}: {confidence:.1f}%\n"

        # Add metadata
        results_text += f"\nBody Part: {body_part}\nTimeline: {timeline}"

        # Generate recommendations based on category
        recommendations = ""
        if top_category == "Highly Dangerous":
            recommendations += "⚠️ IMPORTANT: This analysis suggests a potentially serious condition. Please consult with a dermatologist as soon as possible for proper evaluation and diagnosis."

            if top_subcategory == "Malignant Melanoma":
                recommendations += "\n\nWatch for the ABCDEs of melanoma:\n• Asymmetry\n• Border irregularity\n• Color variations\n• Diameter > 6mm\n• Evolving size, shape, or color"
            elif top_subcategory == "Non-Melanoma Skin Cancer":
                recommendations += "\n\nLook for:\n• Persistent, non-healing sores\n• Reddish patches\n• Shiny bumps or nodules\n• Pink growths with raised edges"
        else:
            recommendations += "This appears to be a less serious condition, but a healthcare provider should still evaluate it properly."

            if top_subcategory == "Benign Lesions":
                recommendations += "\n\nGeneral care:\n• Monitor for changes\n• Protect from sun exposure\n• Avoid irritation"
            elif top_subcategory == "Inflammatory Conditions":
                recommendations += "\n\nGeneral care:\n• Use gentle skincare products\n• Avoid known triggers\n• Keep skin moisturized\n• Consider over-the-counter anti-inflammatory creams"

        return ("", "Analysis complete!", results_text, recommendations)

    except Exception as e:
        print(f"Error in prediction: {e}")
        return (f"Error analyzing image: {str(e)}", "", "", "")



    # if top_condition in dangerous_conditions:
    #     print("\n⚠️ Warning: Potentially malignant lesion detected. Please consult a dermatologist.")
    # else:
    #     if top_prob < 0.60:  # If the model is unsure (low top confidence)
    #         print("\n⚠️ The model is unsure. Please consult a dermatologist for a professional evaluation.")
    #     else:
    #         print("\n👍 Likely a benign condition. Monitor for changes and consult a dermatologist if concerned.")

# --- 5. RUN EVERYTHING ---


In [ ]:
import gradio as gr
import time
import random

# Fake prediction function
def predict_skin_condition(image):
    time.sleep(2)
    return {
        "Dermatitis": random.randint(70, 90),
        "Psoriasis": random.randint(60, 85),
        "Eczema": random.randint(50, 75)
    }

def analyze_image(image, body_part, timeline):
    if random.random() < 0.15:
        return ("Poor image quality detected. Please upload a clearer photo.", "", "", "")

    results = predict_skin_condition(image)
    results_text = "\n".join([f"{cond}: {conf}%" for cond, conf in results.items()])

    # Add metadata (Body part and Timeline) to the results
    results_text += f"\n\nBody Part: {body_part}\nTimeline: {timeline}"

    recommendations = generate_recommendations(results)

    # Add care recommendation based on malignant prediction
    if "Psoriasis" in results and results["Psoriasis"] > 85:  # Example for high risk condition
        recommendations += "\n\n**Important:** If you suspect this is a malignant condition, please reach out to your healthcare provider for further evaluation."

    return ("", "Analyzing complete!", results_text, recommendations)

def generate_recommendations(results):
    tips = []
    if results:
        for condition in results:
            if condition == "Dermatitis":
                tips.append("- Use fragrance-free moisturizers. Avoid harsh soaps.")
            elif condition == "Psoriasis":
                tips.append("- Keep skin moisturized. Consider over-the-counter hydrocortisone.")
            elif condition == "Eczema":
                tips.append("- Use gentle cleansers. Moisturize immediately after bathing.")
    return "\n".join(tips)

# Gradio App
with gr.Blocks(css="""
body {
    margin: 0;
    padding: 0;
    font-family: 'Poppins', sans-serif;
}
.gradio-container {
    background: #F9DADA;
    min-height: 100vh;
    display: block; /* flex causes minor blurring */
    margin: 0 auto;
}

.container-card {
    background: #ffffff;
    border-radius: 20px;
    padding: 40px;
    max-width: 700px;
    width: 90%;
    box-shadow: 0px 8px 24px rgba(0,0,0,0.1);
    text-align: center;
    transform: none;
    image-rendering: auto;
    text-rendering: optimizeLegibility;
    -webkit-font-smoothing: antialiased;
}

h1, h2, h3 {
    color: #005C69; /* teal */
    font-weight: bold;
}
p, label, .text-small {
    color: #554348; /* maroon */
    font-size: 16px;
}
#start-button, #continue-button, #scan-button {
    background-color: #426A5A;
    color: white;
    font-size: 18px;
    font-weight: bold;
    border-radius: 12px;
    padding: 14px 24px;
    transition: 0.3s;
}
#start-button:hover, #continue-button:hover, #scan-button:hover {
    background-color: #2f4c3f;
}
.results-text textarea {
    font-size: 20px;
    font-weight: 500;
    color: #777;
    line-height: 1.5;
    padding: 16px;
    border-radius: 12px;
}

.logo-image img {
    border-radius: 20px;
    padding: 8px;
    background: #FFE4E1;
    box-shadow: 0px 4px 12px rgba(0,0,0,0.1);
    width: 140px;
    margin-bottom: 20px;
}
.text-small {
    font-size: 12px;
    color: #777;
}
""") as app:

    # Main card container
    with gr.Row(elem_classes="container-card"):

        skinscope_logo = gr.Image(value="skinscope-logo.png", elem_classes="logo-image", show_label=False, container=False)

        home_screen = gr.Column(visible=True)
        how_screen = gr.Column(visible=False)
        terms_screen = gr.Column(visible=False)
        upload_screen = gr.Column(visible=False)
        analyzing_screen = gr.Column(visible=False)
        results_screen = gr.Column(visible=False)

        # Home Screen
        with home_screen:
            skinscope_logo
            gr.Markdown("""<h1>Welcome to <span style='color:#005C69;'>SkinScope</span></h1>
<p>Your daily partner for loving every inch of your skin 💖</p>""")
            start_button = gr.Button("Start", elem_id="start-button")

        # How It Works
        with how_screen:
            gr.Markdown("<h2>How It Works</h2>")
            gr.Markdown("Upload your skin photo. We analyze it using AI, suggest conditions, and offer skincare advice.")
            how_continue = gr.Button("Continue", elem_id="continue-button")

        # Terms and Consent
        with terms_screen:
            gr.Markdown("<h2>Terms & Consent</h2>")
            agree_terms = gr.Checkbox(label="I agree to the Terms and Conditions.")
            allow_use = gr.Checkbox(label="I allow my images to be used anonymously to improve SkinScope.")
            terms_continue = gr.Button("Continue to Scan", elem_id="continue-button")

        # Upload Screen
        with upload_screen:
            gr.Markdown("<h2>Upload or Take a Skin Photo</h2>")
            image_input = gr.Image(type="filepath", label="Upload your skin image")
            scan_button = gr.Button("Start Scan", elem_id="scan-button")

            body_part_dropdown = gr.Dropdown(choices=body_part_options, label="Select Body Part", visible=True)
            timeline_dropdown = gr.Dropdown(choices=timeline_options, label="Select Timeline", visible=True)



        # Analyzing Screen
        with analyzing_screen:
            error_message = gr.Textbox(label="Error", visible=False)
            analyzing_message = gr.Markdown(visible=False)
            results_box = gr.Textbox(label="Results", visible=False, lines=8)
            recommendations_box = gr.Textbox(label="Recommendations", visible=False, lines=6)

        # Results Screen
        with results_screen:
            gr.Markdown("<h2>Results</h2>")
            final_results = gr.Textbox(label="Skin Condition Results", lines=6, elem_classes="results-text")
            final_recommendations = gr.Textbox(label="Care Recommendations", lines=6, elem_classes="results-text")
            gr.Markdown("""
<p class='text-small'>
<b>Disclaimer:</b> This app is designed to provide informational insights about common skin conditions based on user-submitted images. The results generated by this app are not a medical diagnosis, and the app is not a medical device regulated by the U.S. Food and Drug Administration (FDA).
The content, recommendations, or visual analyses presented are not a substitute for professional medical advice, diagnosis, or treatment. Always seek the guidance of a qualified health provider with any questions you may have regarding a medical condition. Never disregard professional medical advice or delay seeking care based on information provided by this app.
If you believe you may be experiencing a serious or urgent health issue, contact a licensed healthcare professional or call emergency services immediately.

</p>
""")

        # Button Clicks
        start_button.click(lambda: (gr.update(visible=False), gr.update(visible=True)), outputs=[home_screen, how_screen])
        how_continue.click(lambda: (gr.update(visible=False), gr.update(visible=True)), outputs=[how_screen, terms_screen])
        terms_continue.click(lambda agree, allow: (gr.update(visible=False), gr.update(visible=True)) if agree else (gr.update(visible=True), gr.update(visible=False)), inputs=[agree_terms, allow_use], outputs=[terms_screen, upload_screen])

        scan_button.click(lambda: (gr.update(visible=True), gr.update(visible=True)), outputs=[body_part_dropdown, timeline_dropdown])

        #adjust this, outputs = <- put sense
        scan_button.click(
              fn=analyze_image,
              inputs=[image_input, body_part_dropdown, timeline_dropdown],
              outputs=[error_message, analyzing_message, final_results, final_recommendations]
        )

        # Show dropdowns after scanning
        scan_button.click(lambda: (gr.update(visible=True), gr.update(visible=True)), outputs=[body_part_dropdown, timeline_dropdown])

        scan_button.click(lambda: (gr.update(visible=False), gr.update(visible=False), gr.update(visible=True)), outputs=[analyzing_screen, upload_screen, results_screen])


app.launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9c8bb1032e6d05cd31.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
